# LUBM reference comparison — all three index structures

This is the **reference benchmark** for comparing kermit's three index
structures end-to-end. Every step runs live from this notebook:

1. **Verify** — prove the join engine returns the published LUBM(1, 0)
   cardinalities for all 14 queries *before* timing anything.
2. **Sweep** — one `bench run` across all index structures, algorithms, and
   metrics (insertion time, iteration time, heap space). The LUBM(1, 0)
   dataset is generated on demand and cached.
3. **Analyse** — six figures and two statistical tables.

## The three configurations

`HashTrie` implements `HashTrieIterable`, not `TrieIterable`, so it cannot
run LeapfrogTriejoin — `(HashTrie, HashTriejoin)` is the only valid pairing.
The comparison is therefore between three *system configurations*, not three
structures under one algorithm:

| Configuration | Index structure | Join algorithm |
|---|---|---|
| TreeTrie + LeapfrogTriejoin | pointer-based trie | worst-case-optimal multi-way join |
| ColumnTrie + LeapfrogTriejoin | column-oriented trie | worst-case-optimal multi-way join |
| HashTrie + HashTriejoin | hash-based trie | hash-trie multi-way join |

## Requirements

Run from inside `nix develop` (provides cargo, JDK 8 for the LUBM generator,
and the library path the Python wheels need on NixOS). Launch with:

```sh
cd python/kermit-lab && uv run --with jupyter jupyter lab
```

Design rationale: `docs/specs/2026-06-11-lubm-reference-notebook-design.md`.
Workload reference: `docs/benchmarks/LUBM.md`.

## Phase 0 — Preflight

Locate the repo root, fail loudly if the toolchain is missing, and define
`run_phase`, the streamed-subprocess helper every phase below uses.

In [ ]:
import os, subprocess, shutil, time
from pathlib import Path

# Resolve the repo root by walking up until we find Cargo.toml.
# This makes the notebook robust to wherever the kernel was launched from.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "Cargo.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate Kermit repo root (no Cargo.toml found above CWD)")
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
print(f"CWD: {os.getcwd()}")

# Fail loud upfront if the toolchain is missing — better than a confusing
# error 30 seconds into a phase.
assert shutil.which("cargo"), "cargo not on PATH — run from inside `nix develop`"
assert shutil.which("java"),  "java not on PATH — needed by the LUBM generator"
print("cargo:", shutil.which("cargo"))
print("java: ", shutil.which("java"))


def run_phase(args, label):
    """Run a kermit subprocess with timing and streamed output.

    Each phase invokes `cargo run --release -- bench …` (or `cargo test`)
    and may take several minutes on a cold build. Output is streamed live to
    the notebook so the cell does not appear hung. We keep a rolling buffer
    of the last 40 lines and surface it on non-zero exit before raising
    CalledProcessError.
    """
    print(f"⏳ {label} - this may take several minutes...")
    start = time.monotonic()
    proc = subprocess.Popen(
        args,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    last_lines: list[str] = []
    for line in proc.stdout:
        line = line.rstrip()
        last_lines.append(line)
        if len(last_lines) > 40:
            last_lines.pop(0)
        print(line)
    rc = proc.wait()
    elapsed = time.monotonic() - start
    if rc != 0:
        raise subprocess.CalledProcessError(rc, args, output="\n".join(last_lines))
    print(f"✅ {label} - done in {elapsed:.1f}s")
    return proc

### Sampling profile

`quick` is for live demos (~15–30 min on a cold cache, most of it the one-off
dataset generation and the correctness test). `full` is for thesis-grade
numbers — expect several times longer. Criterion output directories are
shared between profiles, so figures always reflect the most recent sweep;
the report JSON is profile-suffixed so `kl.load` picks up the right run.

In [ ]:
PROFILE = "quick"  # "quick" | "full"

SAMPLING = {
    "quick": ["--sample-size", "10", "--measurement-time", "1", "--warm-up-time", "1"],
    "full":  ["--sample-size", "50", "--measurement-time", "3", "--warm-up-time", "1"],
}[PROFILE]
REPORT_JSON = f"bench-runs/lubm-reference-sweep-{PROFILE}.json"
print(f"profile={PROFILE}")
print(f"sampling flags: {' '.join(SAMPLING)}")
print(f"report → {REPORT_JSON}")

## Phase 1 — Correctness before timing

Timing numbers from an engine that returns wrong results are worthless, so
the first real step is the cardinality oracle:
`kermit/tests/lubm_cardinalities.rs` generates LUBM(1, 0) end-to-end
(vendored UBA jar → N-Triples → Univ-Bench TBox entailment → partition →
parquet), runs all 14 queries through the join engine, and asserts every
result count matches the published reference cardinalities (LUBM paper,
Table 3).

The test deliberately regenerates into a tempdir rather than reusing the
benchmark cache: it validates the *pipeline*, not a cached snapshot. On
notebook reruns you can skip this cell — it is independent of everything
below.

In [ ]:
run_phase(
    ["cargo", "test", "-p", "kermit", "--test", "lubm_cardinalities",
     "--release", "--", "--nocapture"],
    "Phase 1: verify all 14 LUBM(1,0) cardinalities match the paper",
)